# 00 — Data Wrangling for Zebrafish Behavioral Battery

This notebook is the source-data translation layer for the zebrafish behavioral battery project. It prepares the master dataset for the R-based audit, modeling, battery-comparison, and PCA notebooks.

It does **not** run inferential statistics. It standardizes labels, parses datetime metadata, derives planned variables, applies documented protocol/data-quality exclusions, flags ambiguous cases, and exports battery-level CSVs plus machine-readable logs.

Core principle: ambiguous but analyzable observations are retained with flags. Rows are removed globally only when a documented protocol failure, tracking failure, impossible critical value, duplicate identifier, or absence of analyzable data makes the row unsuitable for downstream analysis.

## A. Purpose and scope

This notebook produces the starting battery files for the formal R audit.

| Output | Location |
|---|---|
| BT1 battery dataset | `data_processed/battery_suite_1.csv` |
| BT2 battery dataset | `data_processed/battery_suite_2.csv` |
| BT3 battery dataset | `data_processed/battery_suite_3.csv` |
| Exclusion / retention log | `outputs/logs/exclusion_log.csv` |
| Wrangling QC log | `outputs/logs/wrangling_qc_log.csv` |

Battery definitions:

| Battery | Structure |
|---|---|
| BT1 | Light-dark → Novel tank → Endurance, full sequential battery |
| BT2 | Novel tank → Light-dark, alternative/inverted sequence |
| BT3 | Endurance only, isolated resistance test |

The downstream notebooks decide model families, battery comparisons, and PCA interpretation. This notebook only prepares analyzable columns and explicit logs.

The full-battery PCA later uses the endpoint registry. This wrangling notebook therefore prepares all candidate endpoint columns consistently, including locomotor, light-dark, endurance, glycemia, and condition-factor variables when available.

## B. Imports and project-root path setup

Reusable mechanics live in `wrangling_helpers.py`. The notebook imports those helpers and keeps the visible workflow focused on study-specific decisions.

In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# The helper file is expected either next to this notebook or in the project-level /python folder.
HELPER_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "python",
    Path.cwd().parent,
    Path.cwd().parent / "python",
]

for candidate in HELPER_CANDIDATES:
    if (candidate / "wrangling_helpers.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        HELPER_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find wrangling_helpers.py. Place it next to this notebook "
        "or in the project-level python/ directory."
    )

from wrangling_helpers import (
    add_exclusion_event,
    add_iqr_outlier_flag,
    add_qc_event,
    build_datetime_raw,
    compute_kc,
    compute_mtpc,
    compute_resistance_index,
    ensure_project_paths,
    get_first_match,
    normalize_first_choice,
    parse_datetime_robust,
    safe_divide,
    summarize_battery,
)

paths = ensure_project_paths()
PROJECT_ROOT = paths["PROJECT_ROOT"]
DATA_RAW = paths["DATA_RAW"]
DATA_PROCESSED = paths["DATA_PROCESSED"]
OUTPUTS = paths["OUTPUTS"]
OUTPUT_LOGS = paths["OUTPUT_LOGS"]
OUTPUT_TABLES = paths["OUTPUT_TABLES"]
RAW_MASTER_PATH = paths["RAW_MASTER_PATH"]

print(f"Helper module loaded from: {HELPER_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw input expected at: {RAW_MASTER_PATH}")


Helper module loaded from: C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\python
Project root: C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo
Raw input expected at: C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\data_raw\masters_data.csv


## C. Load raw data

In [2]:
if not RAW_MASTER_PATH.exists():
    raise FileNotFoundError(
        "Raw master dataset not found. Expected file at "
        f"{RAW_MASTER_PATH}. Move the master data file to data_raw/masters_data.csv."
    )

raw = pd.read_csv(RAW_MASTER_PATH)
raw["source_row_index"] = raw.index

print(f"Raw rows: {len(raw):,}")
print(f"Raw columns: {len(raw.columns):,}")
raw.head()

Raw rows: 414
Raw columns: 34


,fish_id,battery_suite,group,date,start_time,treatment,exposure,latency,first_choice,num_changes,time_bright,mtpc,mov_bottom,mov_upper,mov_total,dist_bottom,dist_upper,dist_total,vel_bottom,vel_upper,vel_mean,attempts,last_flux,time_in_last_flux,resistance_index,lt,ls,wt,Kc,sex,blood_sugar,conductivity,pH,source_row_index
0,1BT01hCTR01,1,1h_CTR,2020-07-21,08:52:05,0.0,1,0.0,C,23.0,402.0,17.5,370.0,84.0,454.0,1344.0,399.0,1743.0,3.63,4.75,3.84,2.0,4.0,17.0,7.13,4.2,3.4,0.49,0.00125,NaN,NaN,80.0,7.2,0
1,1BT01hCTR02,1,1h_CTR,2020-07-21,09:09:30,0.0,1,0.0,C,7.0,421.0,60.1,87.0,0.0,87.0,723.1,0.0,723.1,8.31,NaN,8.31,3.0,3.0,17.0,3.85,3.6,2.8,0.37,0.00169,NaN,95.0,80.0,7.2,1
2,1BT01hCTR03,1,1h_CTR,2020-07-21,09:30:15,0.0,1,81.0,C,66.0,202.0,3.1,257.0,172.0,429.0,3492.8,2325.5,5818.3,13.59,13.52,13.56,1.0,9.0,52.0,43.80,4.0,3.2,0.51,0.00156,NaN,NaN,80.0,7.2,2
3,1BT01hCTR04,1,1h_CTR,2020-07-21,09:50:12,0.0,1,0.0,C,52.0,151.0,2.9,262.0,180.0,442.0,2653.0,1832.2,4485.2,10.13,10.18,10.15,1.0,8.0,52.0,34.93,3.3,2.5,0.32,0.00205,NaN,119.0,80.0,7.2,3
4,1BT01hCTR05,1,1h_CTR,2020-07-21,10:10:35,0.0,1,0.0,C,40.0,110.0,2.8,196.0,250.0,446.0,2504.6,3333.7,5838.3,12.78,13.33,13.09,2.0,8.0,27.0,31.60,3.8,3.0,0.44,0.00163,m,197.0,80.0,7.2,4


## D. Preserve raw identifiers and source metadata

In [3]:
draft = raw.copy()

if "fish_id" in draft.columns:
    draft["fish_id"] = draft["fish_id"].astype(str).str.strip()
else:
    raise ValueError("Required column `fish_id` is missing from the raw data.")

if "battery_suite" not in draft.columns:
    raise ValueError("Required column `battery_suite` is missing from the raw data.")

for col in ["group", "first_choice", "date", "start_time", "sex"]:
    if col in draft.columns:
        draft[f"{col}_raw"] = draft[col]

draft["battery"] = draft["battery_suite"].map({1: "BT1", 2: "BT2", 3: "BT3"}).fillna(
    draft["battery_suite"].astype(str)
)

front_cols = [
    "source_row_index", "fish_id", "battery_suite", "battery", "group_raw",
    "date_raw", "start_time_raw", "first_choice_raw"
]
front_cols = [c for c in front_cols if c in draft.columns]
remaining_cols = [c for c in draft.columns if c not in front_cols]
draft = draft[front_cols + remaining_cols]

draft.head()

,source_row_index,fish_id,battery_suite,battery,group_raw,date_raw,start_time_raw,first_choice_raw,group,date,start_time,treatment,exposure,latency,first_choice,num_changes,time_bright,mtpc,mov_bottom,mov_upper,mov_total,dist_bottom,dist_upper,dist_total,vel_bottom,vel_upper,vel_mean,attempts,last_flux,time_in_last_flux,resistance_index,lt,ls,wt,Kc,sex,blood_sugar,conductivity,pH,sex_raw
0,0,1BT01hCTR01,1,BT1,1h_CTR,2020-07-21,08:52:05,C,1h_CTR,2020-07-21,08:52:05,0.0,1,0.0,C,23.0,402.0,17.5,370.0,84.0,454.0,1344.0,399.0,1743.0,3.63,4.75,3.84,2.0,4.0,17.0,7.13,4.2,3.4,0.49,0.00125,NaN,NaN,80.0,7.2,NaN
1,1,1BT01hCTR02,1,BT1,1h_CTR,2020-07-21,09:09:30,C,1h_CTR,2020-07-21,09:09:30,0.0,1,0.0,C,7.0,421.0,60.1,87.0,0.0,87.0,723.1,0.0,723.1,8.31,NaN,8.31,3.0,3.0,17.0,3.85,3.6,2.8,0.37,0.00169,NaN,95.0,80.0,7.2,NaN
2,2,1BT01hCTR03,1,BT1,1h_CTR,2020-07-21,09:30:15,C,1h_CTR,2020-07-21,09:30:15,0.0,1,81.0,C,66.0,202.0,3.1,257.0,172.0,429.0,3492.8,2325.5,5818.3,13.59,13.52,13.56,1.0,9.0,52.0,43.80,4.0,3.2,0.51,0.00156,NaN,NaN,80.0,7.2,NaN
3,3,1BT01hCTR04,1,BT1,1h_CTR,2020-07-21,09:50:12,C,1h_CTR,2020-07-21,09:50:12,0.0,1,0.0,C,52.0,151.0,2.9,262.0,180.0,442.0,2653.0,1832.2,4485.2,10.13,10.18,10.15,1.0,8.0,52.0,34.93,3.3,2.5,0.32,0.00205,NaN,119.0,80.0,7.2,NaN
4,4,1BT01hCTR05,1,BT1,1h_CTR,2020-07-21,10:10:35,C,1h_CTR,2020-07-21,10:10:35,0.0,1,0.0,C,40.0,110.0,2.8,196.0,250.0,446.0,2504.6,3333.7,5838.3,12.78,13.33,13.09,2.0,8.0,27.0,31.60,3.8,3.0,0.44,0.00163,m,197.0,80.0,7.2,m


## E. Standardize labels without discarding information

In [4]:

if "first_choice" in draft.columns:
    draft["first_choice_clean"] = draft["first_choice"].apply(normalize_first_choice)
    draft["first_choice"] = draft["first_choice_clean"]
    draft["first_choice_valid_binary"] = draft["first_choice_clean"].isin(["bright", "dark"])
    draft["first_choice_binary"] = draft["first_choice_clean"].map({"dark": 0, "bright": 1}).astype("Float64")
else:
    draft["first_choice_raw"] = pd.NA
    draft["first_choice_clean"] = pd.NA
    draft["first_choice_valid_binary"] = False
    draft["first_choice_binary"] = pd.NA

if "group" in draft.columns:
    clean_group_stage_1 = {
        "96h_CTR": "96h_0.0", "24h_CTR": "24h_0.0", "01h_CTR": "01h_0.0", "1h_CTR": "01h_0.0",
        "1h_0.5": "01h_0.5", "1h_1.0": "01h_1.0",
    }
    clean_group_stage_2 = {
        "01h_0.0": "01h 0.0%", "01h_0.5": "01h 0.5%", "01h_1.0": "01h 1.0%",
        "24h_0.0": "24h 0.0%", "24h_0.5": "24h 0.5%", "24h_1.0": "24h 1.0%",
        "96h_0.0": "96h 0.0%", "96h_0.5": "96h 0.5%", "96h_1.0": "96h 1.0%",
    }
    draft["group"] = draft["group"].replace(clean_group_stage_1).replace(clean_group_stage_2)

if "sex" in draft.columns:
    draft["sex"] = draft["sex"].replace({"m": "male", "f": "female", "M": "male", "F": "female"})

for col in ["treatment", "exposure"]:
    if col in draft.columns:
        draft[col] = draft[col].astype(str).str.strip().replace({"nan": np.nan})

pd.DataFrame({"count": draft["first_choice_raw"].value_counts(dropna=False)})


,count
first_choice_raw,
C,196
NaN,138
E,80


## F. Parse datetime and audit parsing

In [5]:

dt_audit = parse_datetime_robust(build_datetime_raw(draft))
for col in dt_audit.columns:
    draft[col] = dt_audit[col]

print("Datetime parse summary")
print(draft["datetime_parse_method"].value_counts(dropna=False).to_string())

draft.loc[draft["datetime_parse_failed_flag"], ["source_row_index", "fish_id", "battery", "datetime_raw"]].head(20)


Datetime parse summary
datetime_parse_method
standard    278
failed      136


,source_row_index,fish_id,battery,datetime_raw


## G. Convert numeric columns conservatively

In [6]:
numeric_candidates = [
    "latency", "num_changes", "time_bright",
    "mov_bottom", "mov_upper", "dist_bottom", "dist_upper",
    "attempts", "last_flux", "time_in_last_flux",
    "lt", "ls", "wt", "blood_sugar", "conductivity",
]

for col in numeric_candidates:
    if col in draft.columns:
        draft[col] = pd.to_numeric(draft[col], errors="coerce")

print("Numeric conversion complete. No integer casting or rounding is performed in wrangling.")
print([c for c in numeric_candidates if c in draft.columns])

Numeric conversion complete. No integer casting or rounding is performed in wrangling.
['latency', 'num_changes', 'time_bright', 'mov_bottom', 'mov_upper', 'dist_bottom', 'dist_upper', 'attempts', 'last_flux', 'time_in_last_flux', 'lt', 'ls', 'wt', 'blood_sugar', 'conductivity']


## H. Derived-variable plan

The next step creates the endpoint columns needed by later notebooks. Formulas are imported from `wrangling_helpers.py`; this notebook only applies them to the current dataset.

In [7]:
print("Derived-variable helpers imported from wrangling_helpers.py:")
print("- safe_divide")
print("- compute_mtpc")
print("- compute_resistance_index")
print("- compute_kc")
print("- add_iqr_outlier_flag")

Derived-variable helpers imported from wrangling_helpers.py:
- safe_divide
- compute_mtpc
- compute_resistance_index
- compute_kc
- add_iqr_outlier_flag


## I. Derive behavioral and biometric variables

In [8]:
if {"time_bright", "num_changes"}.issubset(draft.columns):
    draft["mtpc"] = draft.apply(compute_mtpc, axis=1)

if {"mov_bottom", "mov_upper"}.issubset(draft.columns):
    draft["mov_total"] = draft[["mov_bottom", "mov_upper"]].sum(axis=1, min_count=1)

if {"dist_bottom", "dist_upper"}.issubset(draft.columns):
    draft["dist_total"] = draft[["dist_bottom", "dist_upper"]].sum(axis=1, min_count=1)

if {"dist_bottom", "mov_bottom"}.issubset(draft.columns):
    draft["vel_bottom"] = [safe_divide(n, d) for n, d in zip(draft["dist_bottom"], draft["mov_bottom"])]

if {"dist_upper", "mov_upper"}.issubset(draft.columns):
    draft["vel_upper"] = [safe_divide(n, d) for n, d in zip(draft["dist_upper"], draft["mov_upper"])]

if {"dist_total", "mov_total"}.issubset(draft.columns):
    draft["vel_mean"] = [safe_divide(n, d) for n, d in zip(draft["dist_total"], draft["mov_total"])]

if {"mov_upper", "mov_total"}.issubset(draft.columns):
    draft["stratum_pref"] = np.where(
        draft["mov_total"].isna(),
        np.nan,
        np.where(draft["mov_total"] == 0, 0.0, (draft["mov_upper"] / draft["mov_total"]) * 100),
    )

if {"last_flux", "time_in_last_flux"}.issubset(draft.columns):
    draft["resistance_index"] = draft.apply(compute_resistance_index, axis=1)

print("Derived-variable creation complete.")

Derived-variable creation complete.


## J. QC flags before exclusions

In [9]:
draft["ldt_present_flag"] = draft["battery_suite"].isin([1, 2])
draft["ntt_present_flag"] = draft["battery_suite"].isin([1, 2])
draft["endurance_present_flag"] = draft["battery_suite"].isin([1, 3])

draft["first_choice_nonbinary_flag"] = draft["ldt_present_flag"] & ~draft["first_choice_valid_binary"]
draft["no_choice_flag"] = draft["first_choice_nonbinary_flag"]
draft["first_choice_exclusion_reason"] = np.where(
    draft["first_choice_nonbinary_flag"],
    "not_valid_for_binary_first_choice_model; retain unless protocol failure",
    pd.NA,
)

draft["blood_sugar_missing_flag"] = draft["blood_sugar"].isna() if "blood_sugar" in draft.columns else False
draft["sex_missing_flag"] = draft["sex"].isna() if "sex" in draft.columns else False

if "time_bright" in draft.columns:
    draft["ldt_time_bright_gt_480_flag"] = draft["ldt_present_flag"] & draft["time_bright"].notna() & (draft["time_bright"] > 480)
    draft["ldt_time_bright_gt_600_flag"] = draft["ldt_present_flag"] & draft["time_bright"].notna() & (draft["time_bright"] > 600)
else:
    draft["ldt_time_bright_gt_480_flag"] = False
    draft["ldt_time_bright_gt_600_flag"] = False

if "mov_total" in draft.columns:
    draft["ntt_mov_total_gt_480_flag"] = draft["ntt_present_flag"] & draft["mov_total"].notna() & (draft["mov_total"] > 480)
    draft["zero_motion_flag"] = draft["ntt_present_flag"] & draft["mov_total"].notna() & (draft["mov_total"] == 0)
    draft["low_motion_flag"] = draft["zero_motion_flag"]
else:
    draft["ntt_mov_total_gt_480_flag"] = False
    draft["zero_motion_flag"] = False
    draft["low_motion_flag"] = False

nonnegative_cols = [
    "latency", "num_changes", "time_bright", "mov_bottom", "mov_upper", "mov_total",
    "dist_bottom", "dist_upper", "dist_total", "attempts", "last_flux", "time_in_last_flux",
    "lt", "ls", "wt", "blood_sugar", "conductivity", "resistance_index",
]
nonnegative_cols = [c for c in nonnegative_cols if c in draft.columns]
draft["impossible_value_flag"] = False
for col in nonnegative_cols:
    draft["impossible_value_flag"] = draft["impossible_value_flag"] | (draft[col].notna() & (draft[col] < 0))

if {"mov_total", "dist_total"}.issubset(draft.columns):
    draft["tracking_failure_flag"] = draft["ntt_present_flag"] & (
        (draft["mov_total"].isna() & draft["dist_total"].isna()) |
        ((draft["mov_total"] == 0) & (draft["dist_total"] > 0))
    )
else:
    draft["tracking_failure_flag"] = False

critical_cols = [c for c in ["fish_id", "battery_suite", "treatment", "exposure", "group"] if c in draft.columns]
draft["missing_critical_field_flag"] = draft[critical_cols].isna().any(axis=1) if critical_cols else False

draft = add_iqr_outlier_flag(draft, "dist_total", ["battery", "treatment", "exposure"], "distance_outlier_flag")

# Context/presence flags and broad supplementary missingness flags should not
# trigger manual review by themselves. They are retained for audit tables, but
# manual_review_flag should identify rows needing behavioral/QC inspection.
structural_or_noncritical_flags = {
    "ldt_present_flag",
    "ntt_present_flag",
    "endurance_present_flag",
    "global_exclusion_flag",
    "blood_sugar_missing_flag",
    "sex_missing_flag",
    "manual_review_flag",
}

qc_review_flags = [
    c for c in draft.columns
    if c.endswith("_flag") and c not in structural_or_noncritical_flags
]

if qc_review_flags:
    draft["manual_review_flag"] = draft[qc_review_flags].any(axis=1)
else:
    draft["manual_review_flag"] = False

flag_cols = [c for c in draft.columns if c.endswith("_flag")]

print("QC flag counts:")
print(draft[flag_cols].sum().sort_values(ascending=False).to_string())

print("\nManual-review flag counts, excluding structural/noncritical flags:")
print(draft[qc_review_flags + ["manual_review_flag"]].sum().sort_values(ascending=False).to_string())

QC flag counts:
ldt_present_flag               278
ntt_present_flag               278
endurance_present_flag         274
blood_sugar_missing_flag       193
sex_missing_flag                20
manual_review_flag              13
low_motion_flag                  6
zero_motion_flag                 6
distance_outlier_flag            5
tracking_failure_flag            2
first_choice_nonbinary_flag      2
no_choice_flag                   2
ldt_time_bright_gt_480_flag      1
datetime_parse_failed_flag       0
ldt_time_bright_gt_600_flag      0
ntt_mov_total_gt_480_flag        0
impossible_value_flag            0
missing_critical_field_flag      0

Manual-review flag counts, excluding structural/noncritical flags:
manual_review_flag             13
zero_motion_flag                6
low_motion_flag                 6
distance_outlier_flag           5
first_choice_nonbinary_flag     2
tracking_failure_flag           2
no_choice_flag                  2
ldt_time_bright_gt_480_flag     1
datetime_parse

## K. Exclusion, retention, and correction logs

In [10]:

exclusion_columns = [
    "fish_id", "battery", "source_dataset", "row_index_original", "treatment", "exposure", "datetime",
    "endpoint_or_stage", "exclusion_scope", "exclusion_reason", "exclusion_type",
    "is_global_exclusion", "is_endpoint_specific_exclusion", "retained_in_processed_dataset", "notes",
]

exclusion_events = []
qc_events = []

known_global_exclusions = {
    "1BT24h01015": {
        "endpoint_or_stage": "light_dark_sequence",
        "exclusion_reason": "jumped from acclimation/central compartment into bright zone before gate was raised",
        "exclusion_type": "protocol_failure",
        "notes": "Original note: latency NaN; light-dark protocol violated before valid test start.",
    },
    "1BT24h00513": {
        "endpoint_or_stage": "light_dark_sequence",
        "exclusion_reason": "fish died during the light-dark test; behavioral sequence invalid",
        "exclusion_type": "protocol_failure",
        "notes": "Original note: num_changes NaN after death during test.",
    },
    "2BT24h00501": {
        "endpoint_or_stage": "novel_tank_sequence",
        "exclusion_reason": "all novel-tank data missing with no corresponding lab note; BT2 sequence not analyzable",
        "exclusion_type": "missing_critical_field",
        "notes": "Original note: NaN across all novel-tank columns.",
    },
    "3BT01h01015": {
        "endpoint_or_stage": "endurance",
        "exclusion_reason": "all endurance and biometric data missing with no corresponding lab note",
        "exclusion_type": "missing_critical_field",
        "notes": "Original note: no analyzable BT3 data.",
    },
}

for fish_id, meta in known_global_exclusions.items():
    row = get_first_match(draft, fish_id)
    if row is None:
        add_qc_event(qc_events, "known_global_exclusion_id_not_found", fish_id=fish_id, notes=meta["exclusion_reason"])
        continue
    add_exclusion_event(
        exclusion_events,
        row=row,
        endpoint_or_stage=meta["endpoint_or_stage"],
        exclusion_scope="global",
        exclusion_reason=meta["exclusion_reason"],
        exclusion_type=meta["exclusion_type"],
        is_global_exclusion=True,
        is_endpoint_specific_exclusion=False,
        retained_in_processed_dataset=False,
        notes=meta["notes"],
    )

for _, row in draft.loc[draft["first_choice_nonbinary_flag"]].iterrows():
    add_exclusion_event(
        exclusion_events,
        row=row,
        endpoint_or_stage="first_choice_binary",
        exclusion_scope="endpoint_specific",
        exclusion_reason="first_choice is not a valid bright/dark binary observation; retained for other endpoints",
        exclusion_type="no_choice_behavioral",
        is_global_exclusion=False,
        is_endpoint_specific_exclusion=True,
        retained_in_processed_dataset=True,
        notes="Filter only in downstream binary first-choice models or sensitivity analyses.",
    )

for _, row in draft.loc[draft["datetime_parse_failed_flag"]].iterrows():
    add_exclusion_event(
        exclusion_events,
        row=row,
        endpoint_or_stage="metadata_datetime",
        exclusion_scope="flag_only",
        exclusion_reason="datetime could not be parsed; row retained with datetime_raw available",
        exclusion_type="other",
        is_global_exclusion=False,
        is_endpoint_specific_exclusion=False,
        retained_in_processed_dataset=True,
        notes="Check datetime_raw in audit notebook.",
    )

for flag, reason in [
    ("ldt_time_bright_gt_480_flag", "time_bright exceeds 480 seconds; possible scoring-window ambiguity"),
    ("ldt_time_bright_gt_600_flag", "time_bright exceeds 600 seconds; likely impossible if 10-minute LDT window"),
    ("ntt_mov_total_gt_480_flag", "novel-tank movement time exceeds stated 480-second analysis window"),
]:
    if flag in draft.columns:
        for _, row in draft.loc[draft[flag]].iterrows():
            add_exclusion_event(
                exclusion_events,
                row=row,
                endpoint_or_stage="duration_window_qc",
                exclusion_scope="flag_only",
                exclusion_reason=reason,
                exclusion_type="other",
                is_global_exclusion=False,
                is_endpoint_specific_exclusion=False,
                retained_in_processed_dataset=True,
                notes=f"Flag column: {flag}",
            )

exclusion_log = pd.DataFrame(exclusion_events, columns=exclusion_columns)
print(f"Exclusion/retention events logged: {len(exclusion_log)}")
exclusion_log.head(20)


Exclusion/retention events logged: 7


,fish_id,battery,source_dataset,row_index_original,treatment,exposure,datetime,endpoint_or_stage,exclusion_scope,exclusion_reason,exclusion_type,is_global_exclusion,is_endpoint_specific_exclusion,retained_in_processed_dataset,notes
0,1BT24h01015,BT1,masters_data.csv,91,1.0,24,2020-09-29 14:55:25,light_dark_sequence,global,jumped from acclimation/central compartment in...,protocol_failure,True,False,False,Original note: latency NaN; light-dark protoco...
1,1BT24h00513,BT1,masters_data.csv,73,0.5,24,2020-08-12 13:59:52,light_dark_sequence,global,fish died during the light-dark test; behavior...,protocol_failure,True,False,False,Original note: num_changes NaN after death dur...
2,2BT24h00501,BT2,masters_data.csv,202,0.5,24,2021-08-11 10:00:30,novel_tank_sequence,global,all novel-tank data missing with no correspond...,missing_critical_field,True,False,False,Original note: NaN across all novel-tank columns.
3,3BT01h01015,BT3,masters_data.csv,324,1.0,1,NaT,endurance,global,all endurance and biometric data missing with ...,missing_critical_field,True,False,False,Original note: no analyzable BT3 data.
4,1BT01h00516,BT1,masters_data.csv,29,0.5,1,2020-07-22 13:44:30,first_choice_binary,endpoint_specific,first_choice is not a valid bright/dark binary...,no_choice_behavioral,False,True,True,Filter only in downstream binary first-choice ...
5,2BT96h01014,BT2,masters_data.csv,275,1.0,96,2021-08-22 15:31:01,first_choice_binary,endpoint_specific,first_choice is not a valid bright/dark binary...,no_choice_behavioral,False,True,True,Filter only in downstream binary first-choice ...
6,1BT01h00510,BT1,masters_data.csv,23,0.5,1,2020-07-22 11:50:30,duration_window_qc,flag_only,time_bright exceeds 480 seconds; possible scor...,other,False,False,True,Flag column: ldt_time_bright_gt_480_flag


## L. Apply documented global exclusions and manual corrections

In [11]:
global_exclusion_ids = set(known_global_exclusions.keys())
draft["global_exclusion_flag"] = draft["fish_id"].isin(global_exclusion_ids)
draft["global_exclusion_reason"] = draft["fish_id"].map({
    fish_id: meta["exclusion_reason"] for fish_id, meta in known_global_exclusions.items()
})

processed = draft.loc[~draft["global_exclusion_flag"]].copy()

if {"battery_suite", "wt"}.issubset(processed.columns):
    wt_correction_mask = processed["battery_suite"].eq(1) & processed["wt"].notna() & (processed["wt"] > 1)
    for idx, row in processed.loc[wt_correction_mask].iterrows():
        old_value = row["wt"]
        new_value = old_value / 10
        processed.loc[idx, "wt"] = new_value
        add_qc_event(
            qc_events,
            event_type="manual_weight_decimal_correction",
            fish_id=row.get("fish_id"),
            battery=row.get("battery"),
            variable="wt",
            old_value=old_value,
            new_value=new_value,
            notes="BT1 weight > 1 g corrected by dividing by 10 according to documented lab-note rationale.",
        )

processed["Kc"] = compute_kc(processed)

print(f"Rows before global exclusions: {len(draft):,}")
print(f"Rows after global exclusions:  {len(processed):,}")


Rows before global exclusions: 414
Rows after global exclusions:  410


## M. Battery split and final column ordering

In [12]:
core_order = [
    "source_row_index", "fish_id", "battery_suite", "battery", "group_raw", "group",
    "treatment", "exposure", "date_raw", "start_time_raw", "datetime_raw", "datetime",
    "datetime_parse_method", "datetime_parse_failed_flag",
    "first_choice_raw", "first_choice", "first_choice_clean", "first_choice_binary",
    "first_choice_valid_binary", "no_choice_flag", "first_choice_nonbinary_flag",
]
core_order = [c for c in core_order if c in processed.columns]
other_cols = [c for c in processed.columns if c not in core_order]
processed = processed[core_order + other_cols]

bt1 = processed.loc[processed["battery_suite"].eq(1)].copy()
bt2 = processed.loc[processed["battery_suite"].eq(2)].copy()
bt3 = processed.loc[processed["battery_suite"].eq(3)].copy()

print("Battery row counts after documented global exclusions:")
print(pd.Series({"BT1": len(bt1), "BT2": len(bt2), "BT3": len(bt3)}).to_string())

Battery row counts after documented global exclusions:
BT1    136
BT2    139
BT3    135


## N. Export processed datasets and logs

In [13]:
bt1_path = DATA_PROCESSED / "battery_suite_1.csv"
bt2_path = DATA_PROCESSED / "battery_suite_2.csv"
bt3_path = DATA_PROCESSED / "battery_suite_3.csv"
exclusion_log_path = OUTPUT_LOGS / "exclusion_log.csv"
qc_log_path = OUTPUT_LOGS / "wrangling_qc_log.csv"

bt1.to_csv(bt1_path, index=False)
bt2.to_csv(bt2_path, index=False)
bt3.to_csv(bt3_path, index=False)
exclusion_log.to_csv(exclusion_log_path, index=False)

add_qc_event(qc_events, "export_complete", variable="battery_suite_1.csv", new_value=len(bt1), notes=str(bt1_path))
add_qc_event(qc_events, "export_complete", variable="battery_suite_2.csv", new_value=len(bt2), notes=str(bt2_path))
add_qc_event(qc_events, "export_complete", variable="battery_suite_3.csv", new_value=len(bt3), notes=str(bt3_path))
add_qc_event(qc_events, "export_complete", variable="exclusion_log.csv", new_value=len(exclusion_log), notes=str(exclusion_log_path))

qc_log = pd.DataFrame(qc_events)
qc_log.to_csv(qc_log_path, index=False)

print("Exported files:")
for p in [bt1_path, bt2_path, bt3_path, exclusion_log_path, qc_log_path]:
    print(f"- {p}")


Exported files:
- C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\data_processed\battery_suite_1.csv
- C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\data_processed\battery_suite_2.csv
- C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\data_processed\battery_suite_3.csv
- C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\outputs\logs\exclusion_log.csv
- C:\Users\caiqu\OneDrive\Documents\Masters\zebrafish_testbattery_repo\outputs\logs\wrangling_qc_log.csv


## O. Final non-inferential QC summary

In [14]:

summarize_battery(bt1, "BT1 — Light-dark → Novel tank → Endurance")
summarize_battery(bt2, "BT2 — Novel tank → Light-dark")
summarize_battery(bt3, "BT3 — Endurance only")

print("Exclusion/retention event types:")
if len(exclusion_log):
    print(exclusion_log.groupby(["exclusion_scope", "exclusion_type"], dropna=False).size().to_string())
else:
    print("No exclusion/retention events logged.")

print("\nNo inferential conclusions are made in this wrangling notebook.")


BT1 — Light-dark → Novel tank → Endurance
Rows: 136

Rows by group:
group
01h 0.0%    14
01h 0.5%    16
01h 1.0%    16
24h 0.0%    15
24h 0.5%    15
24h 1.0%    14
96h 0.0%    16
96h 0.5%    16
96h 1.0%    14

Rows by treatment × exposure:
treatment  exposure
0.0        1           14
           24          15
           96          16
0.5        1           16
           24          15
           96          16
1.0        1           16
           24          14
           96          14

Missingness in key columns:
blood_sugar    25
sex             9

Active QC flags:
ntt_present_flag               136
ldt_present_flag               136
endurance_present_flag         136
blood_sugar_missing_flag        25
sex_missing_flag                 9
manual_review_flag               8
low_motion_flag                  4
zero_motion_flag                 4
distance_outlier_flag            3
first_choice_nonbinary_flag      1
no_choice_flag                   1
ldt_time_bright_gt_480_flag      1

BT

## P. Completion criteria

This notebook is complete when it has exported:

- three battery-level CSVs in `data_processed/`;
- an exclusion/retention log in `outputs/logs/`;
- a wrangling QC log in `outputs/logs/`;
- a non-inferential summary of row counts, missingness, and active QC flags.

The next notebook, `01_data_audit_and_qc.Rmd`, audits these exported files before any endpoint modeling, battery comparison, or PCA.